In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [ ]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'data_source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3://sports-consolidated-dp/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed'

# DEFINE THE TABLES
bronze_table = f'{catalog}.{bronze_schema}.{data_source}'
silver_table = f'{catalog}.{silver_schema}.{data_source}'
gold_table = f'{catalog}.{gold_schema}.{data_source}'

# Base Path:  s3://sportsbar-final/orders
# Landing Path:  s3://sportsbar-final/orders/landing/
# Processed Path:  s3://sportsbar-final/orders/processed/

In [ ]:
# ---> BRONZE <---
df = spark.read.options(header=True, inferSchema=True).csv(f'{landing_path}/*.csv').withColumn('read_timestamp', F.current_timestamp()).select("*", '_metadata.file_name', '_metadata.file_size')

# 51810
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |FOCT62720602|Tuesday, Septembe...|     ABC987|  25891301|     71.0|2026-11-25 23:58:...|orders_2025_09_30...|    41446|
# |FOCT62720602|Tuesday, Septembe...|     789720|  25891502|    125.0|2026-11-25 23:58:...|orders_2025_09_30...|    41446|
# |FOCT62720602|Tuesday, Septembe...|     789720|  25891403|    462.0|2026-11-25 23:58:...|orders_2025_09_30...|    41446|
# |FOCT62720602|Tuesday, Septembe...|    INVALID|  25891601|    133.0|2026-11-25 23:58:...|orders_2025_09_30...|    41446|
# |FOCT62720602|Tuesday, Septembe...|     789720|  25891602|     79.0|2026-11-25 23:58:...|orders_2025_09_30...|    41446|
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# only showing top 5 rows